#   Bronze to Silver


## Parâmetros do Ambiente - Organização do Ambiente

In [0]:
#PADRÃO: <catalog>.<camada>.<tabela>
catalog = "workspace"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

## Regras Gerais da Camada Silver

- **Nenhuma alteração é feita na camada Bronze.** As tabelas Silver são geradas a partir de leituras das tabelas Bronze já existentes, sem sobrescrever ou modificar os dados originais.
- **Nomes de colunas em português.** Todas as colunas de origem (em inglês) são renomeadas conforme o mapeamento definido para cada tabela.
- **Tipagem correta.** Cada coluna recebe o tipo de dado apropriado (datas, números, texto), corrigindo a tipagem genérica herdada da Bronze.
- **Regras de negócio específicas de cada tabela** (limpeza, deduplicação, tratamento de nulos, etc.) serão aplicadas e comentadas individualmente na seção correspondente a cada tabela, justificando as decisões tomadas.

In [0]:
expected_files = [
    f"{bronze_schema}.tb_movies_info",
    f"{bronze_schema}.tb_movies_metrics",
    f"{bronze_schema}.tb_movies_reviews",
    f"{bronze_schema}.tb_movies_financials",
    f"{bronze_schema}.tb_credits_and_tags",
    f"{bronze_schema}.tb_cotacao_dolar",
]

existing = []
for f in expected_files:
    flag = spark.catalog.tableExists(f)
    if flag:
        existing.append(f)

missing = [f for f in expected_files if f not in existing]
if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
        
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in existing:
        print(f"- {f}")


    # Verificações rápidas antes do Bronze to Silver
    # Antes de seguir para os tratamentos, decidi validar que todas as tabelas Bronze existem, 
    # que a leitura ocorre sem falhas silenciosas e que a coluna "ingestion_datetime" 
    # (usada mais adiante nas regras de deduplicação) está presente em todas elas.
    #Também achei interessante comparar a quantidade total de linhas com o total de linhas únicas. Se essas quantidades forem diferents,
    #então podemos concluir que existem linhas duplicadas no dataframe.
    for str_df in existing:
        df = spark.table(str_df)
        has_data_ingestion_datetime = "Sim" if "ingestion_datetime" in df.columns else "Não"
        has_linhas_duplicadas = "Sim" if df.count() != df.distinct().count() else "Não"
        print(f"\n {str_df.split('.')[-1].upper()}:\n TOTAL LINHAS: {df.count()} linhas.| LINHAS ÚNICAS: {df.distinct().count()} linhas | POSSUI LINHAS DUPLICADAS: {has_linhas_duplicadas} |POSSUI A COLUNA INGESTION DATETIME: {has_data_ingestion_datetime}")
        print("_____________________________________________________________________________________________________________________________________")

# Funções gerais/ reutilizáveis

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
def renomear_colunas(df, dic_renomear):
    for nome_original, novo_nome in dic_renomear.items():
        df = df.withColumnRenamed(nome_original, novo_nome)
    return df

def converter_tipo(df, dic_coluna_tipo):
    for coluna, tipo in dic_coluna_tipo.items():
        df = df.withColumn(coluna, col(coluna).try_cast(tipo))
    return df

def detectar_formatos_data(df, coluna):
    return (
        df
        .withColumn("_formato_detectado", regexp_replace(col(coluna).cast("string"), "[0-9]", "9"))
        .groupBy("_formato_detectado")
        .count()
        .orderBy(col("count").desc())
    )

def investigar_convencao_dia_mes(df, coluna, separador):
    
    padrao_regex = rf"^(\d{{1,2}}){re.escape(separador)}(\d{{1,2}}){re.escape(separador)}(\d{{4}})$"

    df_blocos = (
        df
        .filter(col(coluna).rlike(padrao_regex))
        .select(
            regexp_extract(col(coluna), padrao_regex, 1).cast("int").alias("_bloco1"),
            regexp_extract(col(coluna), padrao_regex, 2).cast("int").alias("_bloco2"),
        )
    )

    qtd_bloco1_maior_12 = df_blocos.filter(col("_bloco1") > 12).count()
    qtd_bloco2_maior_12 = df_blocos.filter(col("_bloco2") > 12).count()

    if qtd_bloco1_maior_12 > 0 and qtd_bloco2_maior_12 == 0:
        convencao = "dd/MM/yyyy"
        observacao = "Evidência só em bloco1 > 12: formato é dia/mês/ano."
    elif qtd_bloco2_maior_12 > 0 and qtd_bloco1_maior_12 == 0:
        convencao = "MM/dd/yyyy"
        observacao = "Evidência só em bloco2 > 12: formato é mês/dia/ano."
    elif qtd_bloco1_maior_12 == 0 and qtd_bloco2_maior_12 == 0:
        convencao = None
        observacao = (
            "Nenhum valor revelador encontrado (todos os blocos <= 12) — "
            "não dá pra confirmar a convenção só pelos dados. Escolha a mais "
            "provável pro contexto e documente a suposição no código."
        )
    else:
        convencao = "dd/MM/yyyy" if qtd_bloco1_maior_12 >= qtd_bloco2_maior_12 else "MM/dd/yyyy"
        observacao = (
            f"ATENÇÃO: evidência nos dois sentidos dentro do mesmo padrão "
            f"'99{separador}99{separador}9999' (bloco1>12: {qtd_bloco1_maior_12} linhas, "
            f"bloco2>12: {qtd_bloco2_maior_12} linhas) — indica mistura real de formatos, "
            f"não só ambiguidade pontual. Convenção escolhida ({convencao}) foi a mais "
            f"frequente; documente essa suposição, os casos ambíguos do meio vão herdá-la."
        )

    print(observacao)

    return {
        "separador": separador,
        "qtd_bloco1_maior_12": qtd_bloco1_maior_12,
        "qtd_bloco2_maior_12": qtd_bloco2_maior_12,
        "convencao_sugerida": convencao,
        "observacao": observacao,
    }

def tratar_coluna_data(df, coluna, formatos_spark, nome_coluna_saida=None):
    nome_coluna_saida = nome_coluna_saida or coluna
    coluna_temp = "_data_convertida_tmp"

    qtd_nulos_antes = df.filter(col(coluna).isNull()).count()
    tentativas_conversao = [
        expr(f"try_to_date(`{coluna}`, '{fmt}')") for fmt in formatos_spark
    ]

    df_convertido = df.withColumn(coluna_temp, coalesce(*tentativas_conversao))

    qtd_nulos_depois = df_convertido.filter(col(coluna_temp).isNull()).count()
    qtd_nulos_novos = qtd_nulos_depois - qtd_nulos_antes

    print(f"Nulos antes da conversão: {qtd_nulos_antes}")
    print(f"Nulos depois da conversão: {qtd_nulos_depois}")
    print(f"Nulos novos (formato ainda não coberto): {qtd_nulos_novos}")

    if qtd_nulos_novos > 0:
        print("Amostra de valores originais que viraram NULL na conversão:")
        (
            df_convertido
            .filter(col(coluna_temp).isNull() & col(coluna).isNotNull())
            .select(coluna)
            .distinct()
            .show(20, truncate=False)
        )

    return df_convertido.drop(coluna).withColumnRenamed(coluna_temp, nome_coluna_saida)

def mapear_valores(df, coluna, dic_mapa, valor_padrao=None):
    chaves = list(dic_mapa.keys())
    expressao = when(col(coluna) == chaves[0], dic_mapa[chaves[0]])
    for chave in chaves[1:]:
        expressao = expressao.when(col(coluna) == chave, dic_mapa[chave])
    return df.withColumn(coluna, expressao.otherwise(valor_padrao))

## 1. silver.tb_avaliacoes_usuarios

**Origem:** `bronze.tb_movies_reviews`

Consolida as avaliações de usuários por filme, com os nomes de coluna traduzidos:

| Coluna Origem | Coluna Destino |
|---|---|
| id | id_filme |
| nome | nome_usuario |
| nota | nota_usuario |
| comentario | comentario_usuario |

**Regras aplicadas:**
- Remoção de registros integralmente duplicados (mesma combinação de filme, usuário, nota e comentário), garantindo unicidade das avaliações.
- Validação da escala de nota permitida (0 a 10): valores fora dessa faixa são descartados e convertidos para NULL.
- Comentários vazios ou compostos apenas por espaços em branco são padronizados com o texto "Sem comentário", em vez de ficarem nulos ou vazios.

In [0]:
from pyspark.sql.functions import col, when, trim

tb_bronze_avaliacoes_usuarios = spark.table(f"{bronze_schema}.tb_movies_reviews")

# Criação de dicionários para renomear e converter os tipos das colunas em uma única passagem.
dic_renomear = {
    "id" : "id_filme",
    "nome" : "nome_usuario",
    "nota" : "nota_usuario",
    "comentario" : "comentario_usuario"
}

dic_coluna_tipo = {
    "nota_usuario" : "double"
}

tb_avaliacoes_usuarios = renomear_colunas(tb_bronze_avaliacoes_usuarios, dic_renomear)
tb_avaliacoes_usuarios = converter_tipo(tb_avaliacoes_usuarios, dic_coluna_tipo)

# Aplicação das regras de negócio definidas na especificação da tb_avaliacoes_usuarios.

# Decidi aplicar o dropDuplicates por último, depois das transformações de
# nota_usuario e comentario_usuario, e não antes.
# Fiz isso pois se a remoção de duplicatas ocorresse antes do tratamento, duas linhas com valores
# originalmente diferentes (mas ambos inválidos) poderiam se tornar idênticas depois da
# limpeza. Por exemplo, duas notas diferentes e fora da faixa 0-10, que ambas viram NULL.
# Se as demais colunas dessas linhas também fossem iguais, elas se tornariam duplicatas
# "novas", criadas pela própria normalização, e não seriam detectadas por um dropDuplicates
# feito antes do tratamento. Por isso, a deduplicação é feita sobre os valores já limpos,
# garantindo que a tabela final realmente não contenha avaliações duplicadas.
tb_avaliacoes_usuarios = (
    tb_avaliacoes_usuarios
    .withColumn("nota_usuario",
                when(((col("nota_usuario") < 0)  | (col("nota_usuario") > 10)) , None)
                .otherwise(col("nota_usuario")))
    
    .withColumn( "comentario_usuario",
                when(col("comentario_usuario").isNull(), "Sem comentário" )
                .when(trim(col("comentario_usuario")) == "", "Sem comentário")
                .otherwise(col("comentario_usuario"))
    )
     .dropDuplicates(["id_filme", "nome_usuario","nota_usuario", "comentario_usuario"])
)

# Checagens de qualidade de dados com o padrão visto nas aulas para confirmar que as regras acima
# realmente produziram o resultado esperado antes da gravação na Silver.
dq_check("tb_avaliacoes_usuarios", "nota_usuario entre 0 e 10", tb_avaliacoes_usuarios, ~((col("nota_usuario") < 0)  | (col("nota_usuario") > 10)))
dq_check("tb_avaliacoes_usuarios", "comentario_usuario não nulo ou vazio", tb_avaliacoes_usuarios, (col("comentario_usuario").isNotNull()) &  (trim(col("comentario_usuario")) != ""))
dq_check_unique("tb_avaliacoes_usuarios", "linhas_unicas", tb_avaliacoes_usuarios, ["id_filme", "nome_usuario","nota_usuario", "comentario_usuario"])


tb_avaliacoes_usuarios.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

display(spark.table(f"{silver_schema}.tb_avaliacoes_usuarios").limit(5))


## 2. silver.tb_info_filmes

**Origem:** `bronze.tb_movies_info`

Consolida as informações principais de cada filme, com os nomes de coluna traduzidos:

| Coluna Origem | Coluna Destino |
|---|---|
| id | id_filme |
| title | titulo |
| original_title | titulo_original |
| release_date | data_lancamento |
| runtime | duracao_minutos |
| original_language | idioma_original |
| status | status_filme |
| overview | sinopse |
| tagline | frase_divulgacao |

**Regras aplicadas:**
- Normalização e tradução do status: antes de traduzir os valores (Released → Lançado, Post Production → Pós-Produção, In Production → Em Produção, Planned → Planejado, Rumored → Rumores, Canceled → Cancelado), a coluna é normalizada (remoção de ruídos, hífens sobressalentes e padronização de caixa). Valores corrompidos ou não mapeáveis são padronizados como "Não Informado".
- Deduplicação por filme: quando há registros duplicados na origem, mantém-se exclusivamente a versão mais recente com base em `ingestion_datetime`.
- Tratamento de data multi-formato: a data de lançamento é convertida testando os diferentes padrões presentes na origem de forma robusta. Apenas valores onde a conversão for estritamente impossível são tratados como NULL.
- Coluna derivada: `ano_lancamento`, extraída de `data_lancamento`.

In [0]:
import re
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, when, trim, regexp_replace, regexp_extract, to_date, coalesce,
    expr, year, initcap, row_number
)

tb_info_filmes = spark.table(f"{bronze_schema}.tb_movies_info")

# Criação de dicionários para renomear e converter os tipos das colunas em uma única passagem.
dic_renomear = {
    "id" : "id_filme",
    "title" : "titulo",
    "original_title" : "titulo_original",
    "release_date" : "data_lancamento",
    "runtime" : "duracao_minutos",
    "original_language" : "idioma_original",
    "status" : "status_filme",
    "overview" : "sinopse",
    "tagline" : "frase_divulgacao"
}

dic_coluna_tipo = {
    "duracao_minutos" : "int"
}

tb_info_filmes = renomear_colunas(tb_info_filmes, dic_renomear)
tb_info_filmes = converter_tipo(tb_info_filmes, dic_coluna_tipo)

#Investigação do formato de data_lancamento.
# As hamadas de diagnóstico, revelam quais "formatos" de data existem na coluna e, pros formatos ambíguos 
# qual é a convenção dia/mês predominante em cada um. 
# O resultado dessas chamadas foi o que usei para definir a lista `formatos_spark`.
detectar_formatos_data(tb_info_filmes, "data_lancamento").display()
investigar_convencao_dia_mes(tb_info_filmes, "data_lancamento", separador="/")
investigar_convencao_dia_mes(tb_info_filmes, "data_lancamento", separador="-")

# Decidi utilizar uma janela para a deduplicação. Ela agrupa por id_filme e ordena pela data de ingestão mais recente primeiro, pra depois pegar só a
# linha 1 de cada grupo (a versão mais nova).
janela_versao_mais_recente = Window.partitionBy("id_filme").orderBy(col("ingestion_datetime").desc())

# Conversão de string pra date testando, em ordem, cada formato encontrado na investigação acima 
tb_info_filmes = tratar_coluna_data(
    tb_info_filmes,
    "data_lancamento",
    formatos_spark=["yyyy-MM-dd", "dd/MM/yyyy", "MM-dd-yyyy"],
)

tb_info_filmes = (
    tb_info_filmes

    # Extrai o ano separadamente pra virar coluna própria (ano_lancamento).
    .withColumn("ano_lancamento", year(col("data_lancamento")))

    #Normalização do status
    # Faz a limpeza antes de traduzir, porque comparar direto com o valor bruto ("Post-Production", " released ", "IN PRODUCTION") nunca bateria.
    .withColumn(
        "status_filme",
        initcap(
            trim(
                regexp_replace(
                    regexp_replace(col("status_filme"), "-", " "),
                    "\\s+", " "
                )
            )
        )
    )

    
    # Agora comparo com a coluna já normalizada da etapa acima.
    .withColumn(
        "status_filme",
        when(col("status_filme") == "Released", "Lançado")
        .when(col("status_filme") == "Post Production", "Pós-Produção")
        .when(col("status_filme") == "In Production", "Em Produção")
        .when(col("status_filme") == "Planned", "Planejado")
        .when(col("status_filme") == "Rumored", "Rumores")
        .when(col("status_filme") == "Canceled", "Cancelado")
        .otherwise("Não Informado")
    )

    # Para essa parte, opitei por não utilizar o dropDuplicatesjá que ele não tem noção de "mais recente" e 
    #apenas descarta duplicatas ficando com uma linha qualquer entre elas.
    # Por isso numero as linhas dentro de cada id_filme pela janela definida e mantenho só a linha 1 de cada grupo, que é garantidamente a versão #mais nova.
    .withColumn("_rn", row_number().over(janela_versao_mais_recente))
    .filter(col("_rn") == 1)
    .drop("_rn")

    #Percebi que a coluna "tconst" não consta no mapeamento oficial de colunas da Silver para esta tabela nem é utilizada em nenhuma tabela da
    #camada Gold ou nas perguntas do Desafio de Analytics. Por isso, optei por removê-la em vez de mantê-la sem uso,
    #já que a regra geral da Silver exige que todas as colunas presentes estejam em português e tenham finalidade definida no pipeline.
    .drop("tconst")
)


#Verifico se a deduplicação está correta
(
    tb_info_filmes
    .groupBy("id_filme")
    .count()
    .filter(col("count") > 1)
    .display()
)

#Verifico se a normalização funcionou
# Só devem aparecer os seis rótulos traduzidos ou "Não Informado".
tb_info_filmes.groupBy("status_filme").count().orderBy(col("count").desc()).display()

#Achei estranho não aparecer nenhum cancelado ou rumor então testei rodar o seguinte comando antes de fazer os tratamentos:
#tb_info_filmes.select("status_filme").distinct().display()
#O resultado confirmou que eles não estão presentes no dataframe desde o início

tb_info_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_info_filmes")
display(spark.table(f"{silver_schema}.tb_info_filmes").limit(5))

## 3. silver.tb_generos

**Origem:** `bronze.tb_credits_and_tags`, coluna `genres`

**Regras aplicadas:**
- A coluna `genres` armazena múltiplos gêneros concatenados em uma única string, com separadores inconsistentes (vírgula e ponto e vírgula misturados). Antes de dividir a coluna, é preciso padronizar para um único separador.
- Após padronizar o separador, a coluna é explodida (`split` + `explode`), fazendo com que cada linha do resultado represente um único gênero por filme (um filme com 3 gêneros gera 3 linhas na Silver, cada uma com o mesmo `id_filme` e um gênero diferente).
- Após a explosão, são removidos resíduos que não pertencem ao domínio de gêneros: strings em branco, textos descritivos que não são nomes de gênero, e valores numéricos deslocados (column shift) vindos de separadores extras na origem.
- Critério de validação do domínio de gêneros (lista de referência vs. heurística própria) é uma decisão de projeto documentada no código, já que o PDF não define essa lista.
- Renomear a coluna resultante para um nome em português (`genero`).

In [0]:
from pyspark.sql.functions import col, trim, split, explode, regexp_replace, upper, when

tb_generos = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# O PDF não definiu a lista que utilizei aqui ebtão pensei em formas de como prosseguir em relação a isso.
# Eu decidi usar o catálogo oficial de gêneros do TMDB (a própria fonte da base) em vez de uma apenas uma heurística genérica, porque uma heurística #correria o risco de aceitar lixo que "parece" texto válido ou rejeitar um gênero legítimo só por ser raro. 
# Com uma lista fechada e comparação em upper, os três tipos de resíduo citados no requisito já saem  automaticamente.
mapa_generos_canonico = {
    "ACTION": "Action",
    "ADVENTURE": "Adventure",
    "ANIMATION": "Animation",
    "COMEDY": "Comedy",
    "CRIME": "Crime",
    "DOCUMENTARY": "Documentary",
    "DRAMA": "Drama",
    "FAMILY": "Family",
    "FANTASY": "Fantasy",
    "HISTORY": "History",
    "HORROR": "Horror",
    "MUSIC": "Music",
    "MYSTERY": "Mystery",
    "ROMANCE": "Romance",
    "SCIENCE FICTION": "Science Fiction",
    "TV MOVIE": "TV Movie",
    "THRILLER": "Thriller",
    "WAR": "War",
    "WESTERN": "Western",
}

# Criação de dicionários para renomear e converter os tipos das colunas em uma única passagem.
dic_renomear = {
    "id" : "id_filme"
}

tb_generos = renomear_colunas(tb_generos, dic_renomear)

tb_generos = (
    tb_generos
    # Padroniza o separador antes do split já que a origem mistura vírgula e ponto e vírgula pra separar múltiplos gêneros na mesma célula
    .withColumn("genres_padronizado", regexp_replace(col("genres"), ";", ","))

   
    .withColumn("genero", explode(split(col("genres_padronizado"), ",")))

    # Só depois de já ter uma linha por gênero é que aplico trim e upper nela pois o soark não permitiu encadear com o  upper e o trim no passo #anterior
    .withColumn("genero", upper(trim(col("genero"))))

    .select("id_filme", "genero")
)

qtd_antes_validacao = tb_generos.count()

# Valida contra o domínio e tudo que não é um gênero reconhecido vira NULL aqui.
tb_generos = mapear_valores(tb_generos, "genero", mapa_generos_canonico, valor_padrao=None)

#Resíduo de verdade: valor que não bateu com nenhum dos 19 gêneros válidos e virou NULL no passo anterior
# Duplicata: um par que é válido, mas está repetido
# Ambos são problemas de origem diferente, e pela minha interpretação apenas o primeiro é sujeira real da coluna
# genres e por isso decidi separar os dois na contagem, em vez de jogar tudo em um mesmo "descartado".
tb_generos_validos = tb_generos.filter(col("genero").isNotNull())
qtd_apos_filtro_dominio = tb_generos_validos.count()

tb_generos = tb_generos_validos.dropDuplicates(["id_filme", "genero"])
qtd_final = tb_generos.count()

qtd_descartada_como_residuo = qtd_antes_validacao - qtd_apos_filtro_dominio
qtd_removida_como_duplicata = qtd_apos_filtro_dominio - qtd_final

print(f"Linhas explodidas (antes de qualquer tratamento): {qtd_antes_validacao}")
print(f"Descartadas por não pertencer ao domínio de gêneros (resíduo real): {qtd_descartada_como_residuo}")
print(f"Removidas por serem duplicata de um par (id_filme, gênero) já válido: {qtd_removida_como_duplicata}")
print(f"Linhas finais na tb_generos: {qtd_final}")

# Verifiquei se realmente sobravam os gêneros do dicionário anterior.
tb_generos.groupBy("genero").count().orderBy(col("count").desc()).display()

tb_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_generos")
display(spark.table(f"{silver_schema}.tb_generos").limit(10))

## 4. silver.tb_pessoas_empresas

**Origem:** `bronze.tb_credits_and_tags`, colunas `cast`, `directors`, `writers`, `production_companies`

**Mapeamento de tipo_entidade:**

| Coluna Origem | tipo_entidade Resultante |
|---|---|
| cast | Ator |
| directors | Diretor |
| writers | Roteirista |
| production_companies | Produtora |

**Regras aplicadas:**
- Cada uma das quatro colunas armazena múltiplos valores delimitados na mesma célula — o mesmo padrão de multi-valor já tratado em `genres`. Cada coluna precisa ser explodida (split + explode) separadamente antes de qualquer unificação.
- Depois de explodidas, as quatro partes são consolidadas numa única tabela (união/`unionByName`), cada uma marcada com o `tipo_entidade` correspondente conforme o mapeamento acima — a tabela final tem uma coluna com o nome da pessoa/empresa e outra com o tipo.
- Padronizar a formatação de capitalização dos nomes (mesma lógica já usada na normalização do `status_filme`).
- Remover registros duplicados na dimensão unificada resultante.
- Assim como em `tb_generos`, é esperado aparecer resíduo de Column Shift/separadores extras (branco, texto fora do domínio, valores deslocados) em pelo menos alguma das quatro colunas — vale reaproveitar o raciocínio já validado lá, documentando a decisão de como distinguir "nome de pessoa/empresa válido" de resíduo.

In [0]:
from functools import reduce
from pyspark.sql.functions import col, trim, split, explode, regexp_replace, initcap, lit, size, lower

tb_bronze_credits_and_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

dic_renomear_credits_and_tags = {"id": "id_filme"}
tb_bronze_credits_and_tags = renomear_colunas(tb_bronze_credits_and_tags, dic_renomear_credits_and_tags)

dic_tipo_entidade = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora",
}

def explodir_multivalor(df, coluna_origem, tipo_entidade_valor):
    return (
        df
        .withColumn("_valor_padronizado", regexp_replace(col(coluna_origem), ";", ","))
        .withColumn("nome_pessoa_empresa", explode(split(col("_valor_padronizado"), ",")))
        .withColumn("nome_pessoa_empresa", trim(col("nome_pessoa_empresa")))
        .withColumn("tipo_entidade", lit(tipo_entidade_valor))
        .select("id_filme", "nome_pessoa_empresa", "tipo_entidade")
    )

# Explode cada uma das quatro colunas separadamente e só depois vou unir tudo.
# cada tipo de entidade tratado isoladamente antes da unificação, pra manter o tipo_entidade correto associado a cada linha.
partes_explodidas = [
    explodir_multivalor(tb_bronze_credits_and_tags, coluna_origem, tipo_entidade_valor)
    for coluna_origem, tipo_entidade_valor in dic_tipo_entidade.items()
]

tb_pessoas_empresas_bruta = reduce(lambda df1, df2: df1.unionByName(df2), partes_explodidas)

qtd_antes_validacao = tb_pessoas_empresas_bruta.count()

tb_pessoas_empresas_bruta = tb_pessoas_empresas_bruta.withColumn(
    "_qtd_palavras", size(split(col("nome_pessoa_empresa"), r"\s+"))
)
#Critérios de validação
# Aqui não existe lista fechada de todo ator, diretor ou produtora do mundo. 
# Por essa razão, precisei pensar wm uma heurística que combina vários sinais, cada um pensado
# a partir de resíduo real encontrado durante a investigação.
#  - vazio: sobra de separador duplicado/solto.
#  - puramente numérico: rating/contagem de votos/ID deslocado de outra coluna (Column Shift).
#  - mais de 6 palavras: fragmento de frase ou até título de filme inteiro vazando da origem
#  - começa com minúscula: normalmente é continuação de texto descritivo, não nome próprio.
criterio_nao_vazio = (col("nome_pessoa_empresa").isNotNull()) & (col("nome_pessoa_empresa") != "")
criterio_nao_numerico = ~col("nome_pessoa_empresa").rlike(r"^-?\d+(\.\d+)?$")
criterio_poucas_palavras = col("_qtd_palavras") <= 6
criterio_nao_comeca_minuscula = ~col("nome_pessoa_empresa").rlike(r"^[a-zà-ÿ]")

# Regra: rejeitar qualquer texto que comece com minúscula, sem exceções.
# Testei liberar termos de palavra única em minúscula, mas não compensou: 
# em um teste com 30 casos, apenas 4 eram nomes reais. 
# O restante eram keywords vazadas ("pianist", "suicide", "hero"). 
# Cheguei a conclusão que manter a regra estrita evita muito mais sujeira do que perde dados válidos.
nomes_estilizados_conhecidos = ["mgk", "j-hope", "will.i.am", "miwa"]
criterio_excecao_nome_estilizado = lower(trim(col("nome_pessoa_empresa"))).isin(nomes_estilizados_conhecidos)

placeholders_ausencia_conhecidos = [
    "", "[]", "n/a", "na", "nenhum", "none", "null", "nan", "unknown", "desconhecido", "n/d",
]
criterio_nao_e_placeholder_ausencia = ~lower(trim(col("nome_pessoa_empresa"))).isin(placeholders_ausencia_conhecidos)

generos_para_deteccao_vazamento = [
    "action", "adventure", "animation", "comedy", "crime", "documentary", "drama", "family",
    "fantasy", "history", "horror", "music", "mystery", "romance", "science fiction",
    "tv movie", "thriller", "war", "western",
]
criterio_nao_e_genero_vazado = ~lower(trim(col("nome_pessoa_empresa"))).isin(generos_para_deteccao_vazamento)

# Ocorreu vazamento do idioma original (original_language) pra dentro de directors e decidi criar essa lista simples quecobre os idiomas mais comuns #do um catálogo
idiomas_comuns_conhecidos = [
    "english", "french", "spanish", "german", "italian", "japanese", "korean", "chinese",
    "mandarin", "cantonese", "hindi", "portuguese", "russian", "arabic", "turkish", "swedish",
    "danish", "norwegian", "dutch", "polish", "czech", "hungarian", "greek", "hebrew", "thai",
    "vietnamese", "indonesian", "filipino", "tagalog", "finnish", "romanian", "ukrainian",
]
criterio_nao_e_idioma_vazado = ~lower(trim(col("nome_pessoa_empresa"))).isin(idiomas_comuns_conhecidos)

criterio_valido = (
    criterio_nao_vazio
    & criterio_nao_numerico
    & criterio_poucas_palavras
    & (criterio_nao_comeca_minuscula | criterio_excecao_nome_estilizado)
    & criterio_nao_e_placeholder_ausencia
    & criterio_nao_e_genero_vazado
    & criterio_nao_e_idioma_vazado
)

qtd_resgatada_pela_excecao = tb_pessoas_empresas_bruta.filter(criterio_excecao_nome_estilizado).count()
print(f"Linhas resgatadas pela lista de exceções de nomes estilizados: {qtd_resgatada_pela_excecao}")

print("Amostra de valores que serão descartados como resíduo:")
(
    tb_pessoas_empresas_bruta
    .filter(~criterio_valido)
    .select("nome_pessoa_empresa", "tipo_entidade")
    .distinct()
    .show(20, truncate=False)
)

tb_pessoas_empresas_validas = tb_pessoas_empresas_bruta.filter(criterio_valido).drop("_qtd_palavras")
qtd_apos_filtro_dominio = tb_pessoas_empresas_validas.count()

tb_pessoas_empresas_validas = tb_pessoas_empresas_validas.withColumn(
    "nome_pessoa_empresa",
    initcap(regexp_replace(trim(col("nome_pessoa_empresa")), r"\s+", " "))
)

tb_pessoas_empresas = tb_pessoas_empresas_validas.dropDuplicates(["id_filme", "nome_pessoa_empresa", "tipo_entidade"])
qtd_final = tb_pessoas_empresas.count()

qtd_descartada_como_residuo = qtd_antes_validacao - qtd_apos_filtro_dominio
qtd_removida_como_duplicata = qtd_apos_filtro_dominio - qtd_final

print(f"Linhas explodidas (antes de qualquer tratamento): {qtd_antes_validacao}")
print(f"Descartadas por não parecerem nome válido (resíduo, heurística): {qtd_descartada_como_residuo}")
print(f"Removidas por serem duplicata do MESMO (id_filme, nome, tipo) já válido: {qtd_removida_como_duplicata}")
print(f"Linhas finais na tb_pessoas_empresas: {qtd_final}")

tb_pessoas_empresas.groupBy("tipo_entidade").count().orderBy(col("count").desc()).display()
tb_pessoas_empresas.groupBy("nome_pessoa_empresa", "tipo_entidade").count().orderBy(col("count").desc()).show(10, truncate=False)

tb_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_pessoas_empresas")
display(spark.table(f"{silver_schema}.tb_pessoas_empresas").limit(10))

## 5. silver.tb_cotacao_dolar

**Origem:** `bronze.tb_cotacao_dolar`

Estrutura o histórico de cotação do dólar (PTAX, extraído via API do Banco Central na camada Bronze) em uma série temporal contínua, dia a dia, servindo de base para a conversão USD → BRL em `tb_financeiro_filmes`.

**Contexto herdado da ingestão (Bronze) — por que 7 dias e por que faltam cotações:**
- A API do Banco Central só publica cotação PTAX nos dias em que o mercado de câmbio efetivamente opera (dias úteis). Finais de semana e feriados não têm pregão, então não existe cotação nova pra esses dias, não é falha da origem nem da ingestão, é ausência estrutural do dado, esperada desde o desenho do pipeline.
- Por isso, na ingestão, a janela de consulta à API foi definida como os últimos 7 dias corridos a partir da data de execução, e não apenas o dia anterior. No pior caso, rodar o job logo após um feriado prolongado emendado com fim de semana, uma janela de 7 dias garante capturar pelo menos uma cotação de dia útil dentro do período, sem depender de acertar exatamente quantos dias não úteis vieram antes. Essa decisão foi tomada já na Bronze.Aqui na Silver ela é o motivo de existirem lacunas (dias sem linha) que precisam ser preenchidas.

**Regras aplicadas:**
- Geração de uma série temporal contínua (todas as datas do período, sem buracos), unindo a essa série as cotações realmente publicadas pela API.
- Preenchimento por Forward Fill: dias sem cotação (finais de semana/feriados) recebem o valor do último dia útil disponível anterior, em vez de ficarem NULL, assim `tb_financeiro_filmes` sempre encontra uma taxa pra converter USD → BRL, independente de a data de lançamento do filme cair ou não em dia útil.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, to_date, sequence, explode, lit, expr, last, row_number,
    min as spark_min, max as spark_max
)

tb_bronze_cotacao_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

dic_renomear_cotacao = {
    "dataHoraCotacao": "data_cotacao",
    "cotacaoCompra": "cotacao_dolar",
}
dic_coluna_tipo_cotacao = {
    "cotacao_dolar": "decimal(18,6)",
}

tb_cotacao_dolar = renomear_colunas(tb_bronze_cotacao_dolar, dic_renomear_cotacao)

# A API devolve a data já com o horário exato da cotação. Como o join com as demais tabelas Silver vai ocorrer em granularidade diária,
# eu converti direto pra date, descartando o horário.
tb_cotacao_dolar = tb_cotacao_dolar.withColumn("data_cotacao", to_date(col("data_cotacao")))
tb_cotacao_dolar = converter_tipo(tb_cotacao_dolar, dic_coluna_tipo_cotacao)


# Como a atividade pede, o job rodará todo dia e a cada execução a Bronze recebe, em
# modo Append, os últimos 7 dias corridos de cotação, não só o dia novo. Essa janela de 7 dias foi
# escolhida propositalmente maior que 1 dia como redundância contra falha de execução. Então, se o Job não
# rodar em um dia, a execução seguinte já recupera sozinha o histórico que ficou pra trás. O efeito
# colateral é que o mesmo dia aparece várias vezes na Bronze ao longo do tempo, uma vez por execução
# em que ele caiu dentro da janela. Por isso, é preciso deduplicar por data_cotacao, mantendo a versão mais recentemente ingerida.
janela_cotacao_mais_recente = Window.partitionBy("data_cotacao").orderBy(col("ingestion_datetime").desc())
tb_cotacao_dolar = (
    tb_cotacao_dolar
    .withColumn("_rn", row_number().over(janela_cotacao_mais_recente))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .select("data_cotacao", "cotacao_dolar")
)


# A tabela acumulada na Bronze só tem linha nos dias em que o mercado de câmbio efetivamente operou finais de semana e feriados não 
# têm dados e por isso nunca vão gerar cotação nova.
# Eu decidi gerar o calendário completo entre o primeiro e o último dia já capturados e unir com as cotações reais via left join, deixando NULL exatamente nos dias sem dados (que serão preenchidos no passo seguinte).
datas_limite = tb_cotacao_dolar.select(
    spark_min("data_cotacao").alias("data_min"),
    spark_max("data_cotacao").alias("data_max"),
).first()

tb_calendario = spark.range(1).select(
    explode(
        sequence(lit(datas_limite["data_min"]), lit(datas_limite["data_max"]), expr("interval 1 day"))
    ).alias("data_cotacao")
)

tb_cotacao_dolar = tb_calendario.join(tb_cotacao_dolar, on="data_cotacao", how="left")

# Dias sem dados recebem a cotação do último dia útil disponível antes dele.
janela_forward_fill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

tb_cotacao_dolar = tb_cotacao_dolar.withColumn(
    "cotacao_dolar",
    last(col("cotacao_dolar"), ignorenulls=True).over(janela_forward_fill)
)


# Depois do forward fill não deve sobrar nenhum dia sem cotação. Identifiquei que a única forma de isso falhar é o primeiro dia do calendário (o #mais antigo capturado na Bronze, já vir nulo, o que não deveria acontecer, já que a API só retorna dias com dados, mas achei interessante confirmar #antes de gravar.
dq_check(
    "tb_cotacao_dolar",
    "cotacao_dolar não nula após forward fill",
    tb_cotacao_dolar,
    col("cotacao_dolar").isNotNull(),
)
dq_check_unique("tb_cotacao_dolar", "linhas_unicas_por_data", tb_cotacao_dolar, ["data_cotacao"])

tb_cotacao_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_cotacao_dolar")
display(spark.table(f"{silver_schema}.tb_cotacao_dolar").orderBy(col("data_cotacao").desc()).limit(10))

## 6. silver.tb_financeiro_filmes

**Origem:** `bronze.tb_movies_financials`

**Mapeamento de colunas:**

| Coluna Origem | Coluna Destino |
|---|---|
| id | id_filme |
| budget | orcamento_usd |
| revenue | receita_usd |

**Regras aplicadas:**
- Deduplicação por `id_filme`, mantendo a versão mais recente por `ingestion_datetime, mesmo padrão de duplicação de Append já visto em `tb_info_filmes`.
- Valores textuais que representam ausência de dado (ex.: "Unknown", "Não Informado", "N/A") são tratados como NULL antes de qualquer higienização ou conversão de tipo, se a limpeza de símbolos rodasse primeiro, esse texto viraria lixo em vez de um NULL identificável.
- Higienização das colunas de orçamento e receita: remoção de símbolos de moeda, separadores de milhar e qualquer caractere que não componha um número, antes da conversão para `decimal(18,2)`.
- Valores zerados ou negativos em orçamento/receita são tratados como ausentes (NULL), não existe filme com orçamento real de R$0, é ausência de dado disfarçada de zero.
- Conversão para Reais (BRL) aplicando a cotação do dólar mais recente disponível em `tb_cotacao_dolar` (a "cotação vigente"), de forma uniforme para todos os filmes, decisão de projeto documentada no código, já que `tb_cotacao_dolar` não cobre o histórico de décadas das datas de lançamento e o pedido de negócio é acompanhar os valores hoje também em reais, não reconstruir uma série histórica de câmbio.
- Colunas derivadas de Lucro (USD e BRL) e Margem de Lucro Percentual, garantindo que a ausência de orçamento ou receita propague corretamente para NULL (em vez de mascarar com um fallback) e que a divisão por orçamento nulo ou zero nunca ocorra.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, when, trim, lower, regexp_replace, row_number, round as spark_round

tb_bronze_financeiro_filmes = spark.table(f"{bronze_schema}.tb_movies_financials")

dic_renomear_financeiro = {
    "id": "id_filme",
    "budget": "orcamento_usd",
    "revenue": "receita_usd",
}
tb_financeiro_filmes = renomear_colunas(tb_bronze_financeiro_filmes, dic_renomear_financeiro)

# Deduplicação
janela_versao_mais_recente = Window.partitionBy("id_filme").orderBy(col("ingestion_datetime").desc())
tb_financeiro_filmes = (
    tb_financeiro_filmes
    .withColumn("_rn", row_number().over(janela_versao_mais_recente))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .select("id_filme", "orcamento_usd", "receita_usd")
)


# orcamento_usd/receita_usd chegam como STRING na Bronze e, além de números "sujos" com símbolo de
# moeda e separador de milhar, também trazem valores textuais de ausência ("Unknown",
# "Não Informado", "N/A" etc.). Preciso tratar esses placeholders antes de tratar o texto e se a
# limpeza de símbolos rodasse primeiro, "Unknown" viraria uma string vazia (ou lixo), em vez de um
# NULL real e identificável.
placeholders_ausencia_financeiro = [
    "unknown", "não informado", "nao informado", "n/a", "na", "nenhum", "none", "null", "nan",
    "desconhecido", "n/d", "-", "",
]

def tratar_placeholder_ausencia(coluna):
    return when(lower(trim(col(coluna))).isin(placeholders_ausencia_financeiro), None).otherwise(col(coluna))

tb_financeiro_filmes = (
    tb_financeiro_filmes
    .withColumn("orcamento_usd", tratar_placeholder_ausencia("orcamento_usd"))
    .withColumn("receita_usd", tratar_placeholder_ausencia("receita_usd"))
)


# Remove qualquer caractere que não seja dígito, ponto ou sinal de menos, símbolo de moeda, separador de milhar e espaços.
#  Só sobra o que realmente compõe um número.
def higienizar_numero(coluna):
    return regexp_replace(col(coluna), r"[^0-9\.\-]", "")

tb_financeiro_filmes = (
    tb_financeiro_filmes
    .withColumn("orcamento_usd", higienizar_numero("orcamento_usd"))
    .withColumn("receita_usd", higienizar_numero("receita_usd"))
)

dic_coluna_tipo_financeiro = {
    "orcamento_usd": "decimal(18,2)",
    "receita_usd": "decimal(18,2)",
}
tb_financeiro_filmes = converter_tipo(tb_financeiro_filmes, dic_coluna_tipo_financeiro)


# Regra de negócio:  orçamento ou receita zerados ou negativos não são valores válidos de negócio, não existe filme com orçamento real de R$0,00 #registrado e por isso tratamos como NULL.

tb_financeiro_filmes = (
    tb_financeiro_filmes
    .withColumn("orcamento_usd", when(col("orcamento_usd") <= 0, None).otherwise(col("orcamento_usd")))
    .withColumn("receita_usd", when(col("receita_usd") <= 0, None).otherwise(col("receita_usd")))
)


# tb_cotacao_dolar só cobre a janela recente que o Job vem acumulando a cada execução, já as muito distante
# das datas de lançamento reais dos filmes (alguns de décadas atrás) não estão sendo extraídas. Por essa razão, não dá pra casar cada filme
# com a cotação do dia exato do seu lançamento com os dados disponíveis na atividade. Como o pedido foi apenas acompanhar os valores hoje 
# também em reais, então eu decidi aplicar a cotação mais recente disponível de forma uniforme a todos os filmes.
tb_cotacao_vigente = (
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .orderBy(col("data_cotacao").desc())
    .limit(1)
    .select(col("cotacao_dolar").alias("cotacao_dolar_vigente"))
)

tb_financeiro_filmes = (
    tb_financeiro_filmes
    .crossJoin(tb_cotacao_vigente)
    .withColumn("orcamento_brl", spark_round(col("orcamento_usd") * col("cotacao_dolar_vigente"), 2))
    .withColumn("receita_brl", spark_round(col("receita_usd") * col("cotacao_dolar_vigente"), 2))
    .drop("cotacao_dolar_vigente")
)

# Lucro e Margem de Lucro
# O llucro não tem significado sem as duas pontas (receita e orçamento), então lucro
# ausente (NULL) é a resposta certa quando falta uma delas, por isso não eu decidir não usar nenhum fallback/coalesce
# nessas duas colunas. Já a margem percentual explicitamente evita dividir por orçamento nulo ou
# zero (o "otherwise" cobre os dois casos), retornando NULL em vez de erro/infinito.
tb_financeiro_filmes = (
    tb_financeiro_filmes
    .withColumn("lucro_usd", col("receita_usd") - col("orcamento_usd"))
    .withColumn("lucro_brl", col("receita_brl") - col("orcamento_brl"))
    .withColumn(
        "margem_lucro_percentual",
        when(
            (col("orcamento_usd").isNotNull()) & (col("orcamento_usd") != 0),
            spark_round((col("lucro_usd") / col("orcamento_usd")) * 100, 2)
        ).otherwise(None)
    )
)

#Check de qualidade
dq_check(
    "tb_financeiro_filmes", "orcamento_usd positivo quando não nulo", tb_financeiro_filmes,
    (col("orcamento_usd").isNull()) | (col("orcamento_usd") > 0)
)
dq_check(
    "tb_financeiro_filmes", "receita_usd positiva quando não nula", tb_financeiro_filmes,
    (col("receita_usd").isNull()) | (col("receita_usd") > 0)
)
dq_check_unique("tb_financeiro_filmes", "linhas_unicas_por_filme", tb_financeiro_filmes, ["id_filme"])

tb_financeiro_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_financeiro_filmes")
display(spark.table(f"{silver_schema}.tb_financeiro_filmes").limit(10))

## 7. silver.tb_metricas_engajamento

**Origem:** `bronze.tb_movies_metrics`

**Mapeamento de colunas:**

| Coluna Origem | Coluna Destino |
|---|---|
| id | id_filme |
| popularity | popularidade |
| vote_average | nota_media_tmdb |
| vote_count | qtd_votos_tmdb |
| averageRating | nota_media_imdb |
| numVotes | qtd_votos_imdb |

**Regras aplicadas:**
- Deduplicação por `id_filme`, mantendo a versão mais recente por `ingestion_datetime', mesmo padrão de duplicação de Append já visto nas demais tabelas Silver.
- A coluna `popularidade` tem inconsistência de separador decimal na origem (ponto vs. vírgula). A formatação é padronizada ANTES da conversão de tipo, pra nenhum valor válido virar NULL só por estar com o separador "errado".
- Column Shift nas colunas de nota e contagem de votos: textos e atributos fora de contexto (vazando de outras colunas) passam por uma conversão de tipo segura, o que não é um número decimal/inteiro válido vira NULL em vez de derrubar o pipeline.
- Notas médias (TMDB e IMDb) fora do intervalo válido de 0 a 10, incluindo valores que parecem ter sido multiplicados por erro de escala (ex.: nota "85" em vez de "8.5"), são descartadas e viram NULL, sem tentativa de correção automática de escala (o PDF pede descartar, não "consertar").
- Contagens de votos (TMDB e IMDb) e o próprio índice de popularidade com valores negativos são invalidados e tratados como NULL, voto negativo ou popularidade negativa não existe, é sinal de dado corrompido.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, when, regexp_replace, row_number

tb_bronze_metricas_engajamento = spark.table(f"{bronze_schema}.tb_movies_metrics")

dic_renomear_metricas = {
    "id": "id_filme",
    "popularity": "popularidade",
    "vote_average": "nota_media_tmdb",
    "vote_count": "qtd_votos_tmdb",
    "averageRating": "nota_media_imdb",
    "numVotes": "qtd_votos_imdb",
}
tb_metricas_engajamento = renomear_colunas(tb_bronze_metricas_engajamento, dic_renomear_metricas)

#Deduplicação
#Bronze acumula registros duplicados por id_filme por causa do Append. Eu numerei as linhas dentro de cada id_filme pela data
# de ingestão mais recente e mantenho só a linha 1 de cada grupo.
janela_versao_mais_recente = Window.partitionBy("id_filme").orderBy(col("ingestion_datetime").desc())
tb_metricas_engajamento = (
    tb_metricas_engajamento
    .withColumn("_rn", row_number().over(janela_versao_mais_recente))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")
)

#Padronização do separador decimal de popularidade
tb_metricas_engajamento = tb_metricas_engajamento.withColumn(
    "popularidade", regexp_replace(col("popularidade").cast("string"), ",", ".")
)

# Conversão de tipo segura
# A base bruta tem deslocamento de colunas, textos e atributos fora de contexto aparecem
# espalhados nas colunas de nota/voto que deveriam ser só numéricas. converter_tipo usa
# try_cast, que devolve NULL pra qualquer valor que não seja um número válido no tipo de
# destino, em vez de estourar erro e derrubar o pipeline.
dic_coluna_tipo_metricas = {
    "popularidade": "double",
    "nota_media_tmdb": "double",
    "qtd_votos_tmdb": "int",
    "nota_media_imdb": "double",
    "qtd_votos_imdb": "int",
}
tb_metricas_engajamento = converter_tipo(tb_metricas_engajamento, dic_coluna_tipo_metricas)


# Nota média (TMDB e IMDb) só faz sentido entre 0 e 10 — qualquer valor fora disso (incluindo os
# que parecem ter vindo multiplicados por erro de escala, tipo "85" no lugar de "8.5") é
# descartado e vira NULL. 
tb_metricas_engajamento = (
    tb_metricas_engajamento
    .withColumn(
        "nota_media_tmdb",
        when((col("nota_media_tmdb") < 0) | (col("nota_media_tmdb") > 10), None).otherwise(col("nota_media_tmdb"))
    )
    .withColumn(
        "nota_media_imdb",
        when((col("nota_media_imdb") < 0) | (col("nota_media_imdb") > 10), None).otherwise(col("nota_media_imdb"))
    )
    .withColumn(
        "qtd_votos_tmdb",
        when(col("qtd_votos_tmdb") < 0, None).otherwise(col("qtd_votos_tmdb"))
    )
    .withColumn(
        "qtd_votos_imdb",
        when(col("qtd_votos_imdb") < 0, None).otherwise(col("qtd_votos_imdb"))
    )
    .withColumn(
        "popularidade",
        when(col("popularidade") < 0, None).otherwise(col("popularidade"))
    )
)

#Checagens de qualidadee
dq_check(
    "tb_metricas_engajamento", "nota_media_tmdb entre 0 e 10", tb_metricas_engajamento,
    (col("nota_media_tmdb").isNull()) | ((col("nota_media_tmdb") >= 0) & (col("nota_media_tmdb") <= 10))
)
dq_check(
    "tb_metricas_engajamento", "nota_media_imdb entre 0 e 10", tb_metricas_engajamento,
    (col("nota_media_imdb").isNull()) | ((col("nota_media_imdb") >= 0) & (col("nota_media_imdb") <= 10))
)
dq_check(
    "tb_metricas_engajamento", "qtd_votos_tmdb não negativa", tb_metricas_engajamento,
    (col("qtd_votos_tmdb").isNull()) | (col("qtd_votos_tmdb") >= 0)
)
dq_check(
    "tb_metricas_engajamento", "qtd_votos_imdb não negativa", tb_metricas_engajamento,
    (col("qtd_votos_imdb").isNull()) | (col("qtd_votos_imdb") >= 0)
)
dq_check(
    "tb_metricas_engajamento", "popularidade não negativa", tb_metricas_engajamento,
    (col("popularidade").isNull()) | (col("popularidade") >= 0)
)
dq_check_unique("tb_metricas_engajamento", "linhas_unicas_por_filme", tb_metricas_engajamento, ["id_filme"])

tb_metricas_engajamento.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_metricas_engajamento")
display(spark.table(f"{silver_schema}.tb_metricas_engajamento").limit(10))